In [ ]:
import pytest
from solution import TextBuffer

def test_initialization():
    """Tests basic buffer initialization."""
    buf_empty = TextBuffer()
    assert len(buf_empty) == 0
    assert str(buf_empty) == ""

    buf_with_text = TextBuffer("hello")
    assert len(buf_with_text) == 5
    assert str(buf_with_text) == "hello"

def test_basic_insert():
    """Tests inserting text at various positions."""
    buf = TextBuffer("world")
    buf.insert(0, "hello ")
    assert str(buf) == "hello world"
    assert len(buf) == 11

    buf.insert(11, "!")
    assert str(buf) == "hello world!"

    buf.insert(6, "cruel ")
    assert str(buf) == "hello cruel world!"

def test_basic_delete():
    """Tests deleting text from various positions."""
    buf = TextBuffer("hello cruel world!")
    buf.delete(6, 6)  # delete "cruel "
    assert str(buf) == "hello world!"
    assert len(buf) == 12

    buf.delete(0, 6)  # delete "hello "
    assert str(buf) == "world!"

    buf.delete(5, 1) # delete "!"
    assert str(buf) == "world"
    assert len(buf) == 5

    buf.delete(0, 5) # delete everything
    assert str(buf) == ""
    assert len(buf) == 0

def test_slicing_and_indexing():
    """Tests __getitem__ for single characters and slices."""
    buf = TextBuffer("0123456789")
    assert buf[0] == "0"
    assert buf[9] == "9"
    assert buf[-1] == "9"
    assert buf[-10] == "0"

    assert buf[2:5] == "234"
    assert buf[:3] == "012"
    assert buf[7:] == "789"
    assert buf[:] == "0123456789"
    assert buf[-5:-2] == "567"

def test_edge_cases_and_errors():
    """Tests boundary conditions and expected exceptions."""
    buf = TextBuffer("abc")

    # Insertions
    with pytest.raises(IndexError):
        buf.insert(4, "!")
    with pytest.raises(IndexError):
        buf.insert(-1, "!")
    buf.insert(3, "d") # Should not raise
    assert str(buf) == "abcd"

    # Deletions
    with pytest.raises(IndexError):
        buf.delete(4, 1)
    with pytest.raises(IndexError):
        buf.delete(0, 5)
    with pytest.raises(IndexError):
        buf.delete(-1, 1)
    buf.delete(3, 1) # Should not raise
    assert str(buf) == "abc"
    
    # Indexing
    with pytest.raises(IndexError):
        _ = buf[3]
    with pytest.raises(IndexError):
        _ = buf[-4]
    
    # Empty buffer operations
    buf_empty = TextBuffer()
    buf_empty.insert(0, "a")
    assert str(buf_empty) == "a"
    buf_empty.delete(0, 1)
    assert str(buf_empty) == ""
    with pytest.raises(IndexError):
        buf_empty.delete(0, 1)

In [ ]:
import pytest
from solution import TextBuffer


    
# --- New tests for Prompt 2 ---

def test_stable_position_creation_and_lookup():
    """Tests that positions can be created and resolved back to indices."""
    buf = TextBuffer("hello")
    pos2 = buf.get_stable_position(2) # Position of 'l'
    assert buf.get_index_for_position(pos2) == 2

    pos_end = buf.get_stable_position(5)
    assert buf.get_index_for_position(pos_end) == 5

def test_position_stability_after_inserts():
    """Tests that positions correctly update after insertions."""
    buf = TextBuffer("world")
    pos_r = buf.get_stable_position(2) # 'r'

    # Insert before
    buf.insert(0, "hello ") # "hello world"
    assert buf.get_index_for_position(pos_r) == 8 # 'r' is now at index 8

    # Insert after
    buf.insert(11, "!") # "hello world!"
    assert buf.get_index_for_position(pos_r) == 8 # 'r' is still at 8

    # Insert at the exact position
    buf.insert(8, "_") # "hello wo_rld!"
    # The original character 'r' is now at index 9
    assert buf.get_index_for_position(pos_r) == 9

def test_position_stability_after_deletes():
    """Tests that positions correctly update after deletions."""
    buf = TextBuffer("hello world")
    pos_w = buf.get_stable_position(6) # 'w'

    # Delete before
    buf.delete(0, 6) # "world"
    assert buf.get_index_for_position(pos_w) == 0 # 'w' is now at index 0

    # Delete after
    buf.delete(4, 1) # "word"
    assert buf.get_index_for_position(pos_w) == 0 # 'w' is still at 0

def test_position_at_deleted_index():
    """Tests resolving a position whose original character was deleted."""
    buf = TextBuffer("abcde")
    pos_c = buf.get_stable_position(2) # 'c'
    
    buf.delete(2, 1) # "abde"
    # The index should now point to 'd', which is the character following 'c'
    assert buf.get_index_for_position(pos_c) == 2

    buf = TextBuffer("abcde")
    pos_e = buf.get_stable_position(4) # 'e'
    buf.delete(4,1) # "abcd"
    # 'e' was last, the index should be the new length of the buffer
    assert buf.get_index_for_position(pos_e) == 4

def test_position_comparison():
    """Tests that positions are ordered correctly."""
    buf = TextBuffer("0123456789")
    positions = [buf.get_stable_position(i) for i in range(10)]
    
    # Check direct comparisons
    assert positions[2] < positions[5]
    assert positions[8] > positions[0]
    
    # Check equality
    pos3_a = buf.get_stable_position(3)
    pos3_b = buf.get_stable_position(3)
    assert pos3_a == pos3_b
    assert not (pos3_a < pos3_b) and not (pos3_b < pos3_a)

    # Check sorting
    import random
    random.shuffle(positions)
    positions.sort()
    
    indices = [buf.get_index_for_position(p) for p in positions]
    assert indices == list(range(10))

def test_position_errors():
    """Tests error handling for invalid indices and positions."""
    buf = TextBuffer("abc")
    with pytest.raises(IndexError):
        buf.get_stable_position(4)
    with pytest.raises(IndexError):
        buf.get_stable_position(-1)

    # Create a valid position, then an invalid one
    pos_b = buf.get_stable_position(1)
    invalid_pos = "not a real position object"
    
    with pytest.raises(ValueError):
        buf.get_index_for_position(invalid_pos)
    with pytest.raises(ValueError):
        buf.get_index_for_position(None)

    # Test position from another buffer
    buf2 = TextBuffer("xyz")
    pos_y = buf2.get_stable_position(1)
    with pytest.raises(ValueError):
        buf.get_index_for_position(pos_y)

In [ ]:
import pytest
from solution import Replica

def test_local_operations_and_text_retrieval():
    """Tests that local operations modify the text as expected."""
    r = Replica("A")
    assert r.get_text() == ""
    
    op1 = r.local_insert(0, "hello")
    assert r.get_text() == "hello"
    
    op2 = r.local_insert(5, " world")
    assert r.get_text() == "hello world"
    
    op3 = r.local_delete(5, 1) # delete space
    assert r.get_text() == "helloworld"
    
    # Check that operations are not None and are distinct
    assert op1 is not None
    assert op2 is not None
    assert op3 is not None
    assert op1 != op2

def test_remote_operation_application():
    """Tests applying an operation from another (conceptual) replica."""
    r1 = Replica("A", "hello")
    r2 = Replica("B", "hello")
    
    op = r1.local_insert(5, " world")
    
    # r2 has not seen the op yet
    assert r2.get_text() == "hello"
    
    r2.apply_remote(op)
    # Now r2 should be up to date
    assert r2.get_text() == "hello world"

def test_idempotency():
    """Tests that applying the same remote operation twice has no effect."""
    r1 = Replica("A", "abc")
    r2 = Replica("B", "abc")
    
    op = r1.local_insert(1, "x") # "axbc"
    
    r2.apply_remote(op)
    assert r2.get_text() == "axbc"
    
    # Apply the same op again
    r2.apply_remote(op)
    assert r2.get_text() == "axbc"

def test_simple_convergence_insert():
    """Tests convergence with concurrent inserts at different positions."""
    r1 = Replica("A", "ab")
    r2 = Replica("B", "ab")

    op1 = r1.local_insert(0, "x") # r1 is "xab"
    op2 = r2.local_insert(2, "y") # r2 is "aby"
    
    assert r1.get_text() == "xab"
    assert r2.get_text() == "aby"

    r1.apply_remote(op2)
    r2.apply_remote(op1)
    
    # Both should converge to the same state
    assert r1.get_text() == r2.get_text()
    # The exact state depends on the CRDT algorithm, but it must be consistent.
    # Possible outcomes: "xaby" or "xyab". The test accepts either.
    assert r1.get_text() in ("xaby", "xyab")


def test_simple_convergence_insert_delete():
    """Tests convergence with a concurrent insert and delete."""
    r1 = Replica("A", "abc")
    r2 = Replica("B", "abc")
    
    op1 = r1.local_delete(1, 1) # r1 is "ac"
    op2 = r2.local_insert(2, "x") # r2 is "abxc"
    
    r1.apply_remote(op2)
    r2.apply_remote(op1)
    
    assert r1.get_text() == r2.get_text()
    assert r1.get_text() == "axc"

def test_concurrent_insert_at_same_position():
    """Tests convergence when two replicas insert at the same position."""
    r1 = Replica("A", "ac")
    r2 = Replica("B", "ac")

    # Both insert at index 1
    op1 = r1.local_insert(1, "b") # r1 is "abc"
    op2 = r2.local_insert(1, "x") # r2 is "axc"
    
    r1.apply_remote(op2)
    r2.apply_remote(op1)
    
    # They must converge to the same text. The tie-breaking (based on replica ID)
    # should be deterministic.
    assert r1.get_text() == r2.get_text()
    
    # To make the test deterministic, we check both possible outcomes.
    # A robust implementation will always pick one based on replica IDs.
    possible_outcomes = {"abxc", "axbc"}
    assert r1.get_text() in possible_outcomes

    # Verify the tie-breaking is consistent
    r3 = Replica("A", "ac")
    r4 = Replica("B", "ac")
    op3 = r3.local_insert(1, "b")
    op4 = r4.local_insert(1, "x")
    r4.apply_remote(op3) # Apply in reverse order
    r3.apply_remote(op4)
    assert r3.get_text() == r4.get_text()
    assert r3.get_text() == r1.get_text() # Must be same outcome as before

In [ ]:
import pytest
from solution import Replica

# --- Basic convergence tests from Prompt 3 ---
def test_simple_convergence_insert():
    r1 = Replica("A", "ab")
    r2 = Replica("B", "ab")
    op1 = r1.local_insert(0, "x")
    op2 = r2.local_insert(2, "y")
    r1.apply_remote(op2)
    r2.apply_remote(op1)
    assert r1.get_text() == r2.get_text()

# --- New, more adversarial tests for Prompt 4 ---

def test_concurrent_delete_and_insert_at_same_position():
    """
    Hard case: R1 deletes a character while R2 inserts right after it.
    The insert should not bring the deleted character back.
    Initial: "abc"
    R1: delete 'b' -> "ac"
    R2: insert 'x' after 'b' (at index 2) -> "abxc"
    Converged: "axc"
    """
    r1 = Replica("A", "abc")
    r2 = Replica("B", "abc")

    op_del = r1.local_delete(1, 1) # 'b'
    op_ins = r2.local_insert(2, "x") # after 'b'
    
    assert r1.get_text() == "ac"
    assert r2.get_text() == "abxc"

    r1.apply_remote(op_ins)
    r2.apply_remote(op_del)
    
    assert r1.get_text() == "axc"
    assert r2.get_text() == "axc"


def test_concurrent_delete_of_same_character_range():
    """
    Both replicas delete the same character. Should be a clean delete.
    Initial: "abcde"
    R1: delete 'bcd' -> "ae"
    R2: delete 'bcd' -> "ae"
    Converged: "ae"
    """
    r1 = Replica("A", "abcde")
    r2 = Replica("B", "abcde")
    
    op1 = r1.local_delete(1, 3)
    op2 = r2.local_delete(1, 3)
    
    assert r1.get_text() == "ae"
    assert r2.get_text() == "ae"
    
    r1.apply_remote(op2)
    r2.apply_remote(op1)
    
    assert r1.get_text() == "ae"
    assert r2.get_text() == "ae"

def test_interleaved_insert_and_delete_anomaly():
    """
    The "resurrection" bug.
    1. R1 inserts 'x' at index 1 -> "axc"
    2. R2 receives op1, applies it -> "axc"
    3. R1 deletes 'x' at index 1 -> "ac"
    4. R2 concurrently inserts 'y' at index 1 (of "ac") -> "ayc"
    5. R1 receives op2, R2 receives op1.
    Converged: "ayc". 'x' must not reappear.
    """
    r1 = Replica("R1", "ac")
    r2 = Replica("R2", "ac")

    # R1 inserts 'x', R2 receives it
    op_ins_x = r1.local_insert(1, "x")
    r2.apply_remote(op_ins_x)
    assert r1.get_text() == "axc"
    assert r2.get_text() == "axc"
    
    # R1 deletes 'x', while R2 concurrently inserts 'y' at the same original position
    op_del_x = r1.local_delete(1, 1)
    op_ins_y = r2.local_insert(1, "y") # in r2's view, this is "ayxc"

    assert r1.get_text() == "ac"
    assert r2.get_text() == "ayxc"
    
    # Apply remote ops
    r1.apply_remote(op_ins_y)
    r2.apply_remote(op_del_x)

    # Check for convergence and that 'x' is gone for good
    assert r1.get_text() == r2.get_text()
    assert "x" not in r1.get_text()
    # The final order of 'a' and 'y' depends on tie-breaking
    assert r1.get_text() in ("ayc", "yac")

def test_causality_with_three_replicas():
    """
    Ensures causality is respected in a more complex graph.
    R1 -> R2 -> R3. An op from R1 must be seen by R3 after R2's op.
    1. R1: "a" -> inserts "b" -> "ab" (op1)
    2. R2 gets op1 -> "ab" -> inserts "c" -> "abc" (op2)
    3. R3 gets op2 from R2. A naive implementation might fail if it hasn't seen op1.
       A correct one should handle this gracefully (e.g., buffer the op).
    4. R3 gets op1 from R1. Now it can apply op2.
    """
    r1 = Replica("R1", "a")
    r2 = Replica("R2", "a")
    r3 = Replica("R3", "a")
    
    # 1. R1 creates an operation
    op1_ins_b = r1.local_insert(1, "b")
    
    # 2. R2 receives it and creates a dependent operation
    r2.apply_remote(op1_ins_b)
    op2_ins_c = r2.local_insert(2, "c")
    
    # 3. R3 receives op2 first. It should either buffer it or handle it
    # without crashing. Its text should not change yet if buffering.
    initial_text = r3.get_text()
    r3.apply_remote(op2_ins_c)
    # Depending on implementation (buffering vs. inert), text may or may not change yet
    
    # 4. R3 now receives the prerequisite operation
    r3.apply_remote(op1_ins_b)
    
    # After receiving both, the final state must be correct.
    assert r3.get_text() == "abc"

In [ ]:
import pytest
from solution import Replica

# --- Basic functionality check ---
r = Replica("A", "hello")
op = r.local_insert(5, " world")
assert r.get_text() == "hello world"

# --- New tests for Prompt 5 ---

def test_add_and_get_marker():
    """Tests basic creation and retrieval of markers."""
    r = Replica("A", "01234")
    marker_id = r.add_marker(2, 'left')
    assert r.get_marker_index(marker_id) == 2
    
    # Test getting an invalid marker
    with pytest.raises(KeyError):
        r.get_marker_index(999)

def test_marker_movement_local_insert():
    """Tests how markers move after local insertions."""
    r = Replica("A", "abc")
    m_left = r.add_marker(1, 'left') # at 'b'
    m_right = r.add_marker(1, 'right') # at 'b'

    # Insert before the markers
    r.local_insert(0, "x") # "xabc"
    assert r.get_marker_index(m_left) == 2
    assert r.get_marker_index(m_right) == 2

    # Insert after the markers
    r.local_insert(4, "y") # "xabcy"
    assert r.get_marker_index(m_left) == 2
    assert r.get_marker_index(m_right) == 2
    
    # Insert at the marker position
    r.local_insert(2, "z") # "xazbc"
    assert r.get_marker_index(m_left) == 2  # 'left' affinity stays put
    assert r.get_marker_index(m_right) == 3 # 'right' affinity moves after 'z'

def test_marker_movement_local_delete():
    """Tests how markers move after local deletions."""
    r = Replica("A", "0123456")
    m = r.add_marker(4, 'left') # at '4'

    # Delete before the marker
    r.local_delete(0, 2) # "23456"
    assert r.get_marker_index(m) == 2

    # Delete after the marker
    r.local_delete(4, 1) # "2346"
    assert r.get_marker_index(m) == 2
    
    # Delete over the marker
    r.local_delete(1, 3) # "26"
    assert r.get_marker_index(m) == 1 # Marker moves to successor

def test_marker_movement_remote_ops():
    """Tests marker stability when remote operations are applied."""
    r1 = Replica("R1", "abc")
    r2 = Replica("R2", "abc")
    
    m1 = r1.add_marker(1, 'right') # r1's cursor on 'b'

    # r2 inserts text before the marker
    op_ins = r2.local_insert(0, "xyz") # r2 is "xyzabc"
    
    # r1 applies the change
    r1.apply_remote(op_ins)
    
    # Marker should have moved
    assert r1.get_marker_index(m1) == 4

    # r2 deletes text including the original marker position
    op_del = r2.local_delete(3, 2) # r2 deletes "ab" -> "xyzc"
    
    r1.apply_remote(op_del)
    # The marker at 'b' should now be at 'c'
    # 'c' is now at index 3 in "xyzc", but it was at 4 in r1's state
    # Let's re-verify the text
    assert r1.get_text() == "xyzc"
    assert r1.get_marker_index(m1) == 3 # The new position of 'c'

def test_delete_marker():
    """Tests that markers can be successfully deleted."""
    r = Replica("A", "abc")
    m_id = r.add_marker(1, 'left')
    assert r.get_marker_index(m_id) == 1
    
    r.delete_marker(m_id)
    
    with pytest.raises(KeyError):
        r.get_marker_index(m_id)

In [ ]:
import pytest
from solution import Replica

def test_simple_undo_redo():
    """Tests a single undo and redo operation."""
    r = Replica("A", "hello")
    r.local_insert(5, " world")
    assert r.get_text() == "hello world"

    r.undo()
    assert r.get_text() == "hello"

    r.redo()
    assert r.get_text() == "hello world"

def test_undo_redo_delete():
    """Tests undoing and redoing a deletion."""
    r = Replica("A", "hello world")
    r.local_delete(5, 6)
    assert r.get_text() == "hello"

    r.undo()
    assert r.get_text() == "hello world"
    
    r.redo()
    assert r.get_text() == "hello"

def test_undo_redo_stack_behavior():
    """Tests multiple undos and the behavior of the redo stack."""
    r = Replica("A")
    r.local_insert(0, "a")
    r.local_insert(1, "b")
    r.local_insert(2, "c")
    assert r.get_text() == "abc"

    r.undo()
    assert r.get_text() == "ab"
    
    r.undo()
    assert r.get_text() == "a"
    
    # New edit should clear the redo stack
    r.local_insert(1, "x")
    assert r.get_text() == "ax"
    
    # Redo should do nothing now
    r.redo()
    assert r.get_text() == "ax"
    
    r.undo()
    assert r.get_text() == "a"
    
    r.redo()
    assert r.get_text() == "ax"

def test_empty_undo_redo_stack():
    """Tests that calling undo/redo on empty stacks does nothing."""
    r = Replica("A", "hello")
    
    # Undo stack is empty
    r.undo()
    assert r.get_text() == "hello"
    
    # Redo stack is empty
    r.redo()
    assert r.get_text() == "hello"
    
    r.local_insert(5, " world")
    r.undo()
    r.redo()
    
    # Redo stack should now be empty
    r.redo()
    assert r.get_text() == "hello world"

def test_undo_with_interleaved_remote_ops():
    """
    The core test for logical undo. The undo should reverse the local
    intent, not just the literal text change.
    """
    r1 = Replica("R1", "ac")
    r2 = Replica("R2", "ac")
    
    # 1. R1 inserts 'b'
    r1.local_insert(1, "b")
    assert r1.get_text() == "abc"
    
    # 2. R2 concurrently inserts 'x'
    op_x = r2.local_insert(1, "x") # R2 is "axc"
    
    # 3. R1 receives R2's change
    r1.apply_remote(op_x)
    
    # R1's text is now "abxc" or "axbc", depending on tie-breaking
    final_text_before_undo = r1.get_text()
    assert final_text_before_undo in ("abxc", "axbc")
    
    # 4. R1 decides to undo the insertion of 'b'
    r1.undo()
    
    # The result should be "axc", regardless of the tie-break.
    # The intent was "ac" + "x", so the 'b' should be gone.
    assert r1.get_text() == "axc"
    
    # 5. Let's test redo as well
    r1.redo()
    assert r1.get_text() == final_text_before_undo

In [ ]:
import pytest
from solution import TextBuffer, BufferInvariantError

def test_invariant_error_not_raised_on_normal_usage():
    """
    The invariant error should not be triggered by typical, efficient workloads.
    """
    try:
        buf = TextBuffer()
        buf.insert(0, "a" * 1000)
        buf.insert(500, "b" * 500)
        buf.delete(200, 100)
        buf.delete(0, 100)
        buf.delete(len(buf)-100, 100)
        buf._check_invariants() # Should pass
        
        buf2 = TextBuffer("hello world")
        buf2._check_invariants() # Should pass
        
        buf3 = TextBuffer()
        buf3._check_invariants() # Should pass
    except BufferInvariantError:
        pytest.fail("BufferInvariantError raised on normal, efficient usage.")

def test_invariant_error_on_pathological_fragmentation():
    """
    This test attempts to create a highly fragmented buffer, which should
    cause _check_invariants() to fail in a naive implementation (e.g. a simple
    list of strings/piece table that doesn't merge). A good implementation
    (e.g. a balanced rope) should handle this and not raise.
    
    The test is designed to probe the implementation quality. If this test
    fails for a submission, it implies the underlying data structure is not robust.
    """
    buf = TextBuffer()
    try:
        # Create thousands of tiny, non-contiguous pieces.
        # A naive piece table would create 1000 pieces.
        # A good rope would rebalance into a shallow tree.
        for i in range(1000):
            buf.insert(i, str(i % 10))
        
        # Now, perform many single-character edits that further fragment the buffer.
        for i in range(500):
            buf.delete(i * 2, 1)
            buf.insert(i * 2, "x")
            
        # At this point, a naive implementation is likely very fragmented.
        # A robust one should have merged or rebalanced itself.
        buf._check_invariants()

    except BufferInvariantError:
        # This is the "expected" failure for a naive implementation.
        # We catch it and pass, assuming this indicates a check is working.
        # A truly excellent implementation will not raise, and the test will
        # also pass. This test distinguishes "working" from "robust".
        pass
    except Exception as e:
        pytest.fail(f"An unexpected error occurred: {e}")

def test_invariant_error_exists_and_is_exception():
    """Checks that the custom error exists and is a proper exception."""
    assert issubclass(BufferInvariantError, Exception)
    with pytest.raises(BufferInvariantError):
        raise BufferInvariantError("test")

In [ ]:
import pytest
from solution import Replica

def is_json_serializable(obj):
    """Helper to check if an object is composed of basic serializable types."""
    if isinstance(obj, (dict, list, str, int, float, bool, type(None))):
        if isinstance(obj, dict):
            return all(isinstance(k, str) and is_json_serializable(v) for k, v in obj.items())
        if isinstance(obj, list):
            return all(is_json_serializable(elem) for elem in obj)
        return True
    return False

def test_snapshot_creation_and_format():
    """Tests that a snapshot can be created and is in a serializable format."""
    r = Replica("A", "hello")
    r.local_insert(5, " world")
    r.local_delete(0, 6)
    
    snapshot = r.get_snapshot()
    
    assert isinstance(snapshot, dict)
    assert is_json_serializable(snapshot), "Snapshot contains non-serializable types"

def test_replica_creation_from_snapshot():
    """Tests creating a new replica from a snapshot and verifies its state."""
    r1 = Replica("R1", "initial text")
    r1.local_insert(0, "a ")
    r1.local_delete(8, 5)
    r1.local_insert(len(r1.get_text()), " z")
    
    snapshot = r1.get_snapshot()
    
    # Create a new replica with a different ID from the snapshot
    r2 = Replica.from_snapshot("R2", snapshot)
    
    assert r2.get_text() == r1.get_text()

def test_convergence_after_loading_from_snapshot():
    """
    The most critical test: ensures a replica loaded from a snapshot can
    correctly integrate new operations and converge with the original replica.
    """
    r1 = Replica("R1", "start")
    r2 = Replica("R2", "start")
    
    # A few initial operations to build up some history
    op1 = r1.local_insert(len(r1.get_text()), " one")
    r2.apply_remote(op1)
    op2 = r2.local_delete(0, 5)
    r1.apply_remote(op2)
    
    assert r1.get_text() == r2.get_text() == " one"
    
    # Now, create a snapshot from r1 and load it into a new replica r3
    snapshot = r1.get_snapshot()
    r3 = Replica.from_snapshot("R3", snapshot)
    
    assert r3.get_text() == r1.get_text()

    # Now, perform new concurrent operations on r1 and r2
    op3 = r1.local_insert(0, "numero ") # r1: "numero one"
    op4 = r2.local_insert(len(r2.get_text()), " plus") # r2: " one plus"
    
    # Apply these new operations to all three replicas
    # r1 needs op4
    r1.apply_remote(op4)
    
    # r2 needs op3
    r2.apply_remote(op3)
    
    # The new replica r3 needs both op3 and op4
    r3.apply_remote(op3)
    r3.apply_remote(op4)
    
    # All three must converge to the exact same final state
    assert r1.get_text() == r2.get_text()
    assert r2.get_text() == r3.get_text()
    
    # The final state should contain both "numero " and " plus"
    assert "numero" in r1.get_text()
    assert "plus" in r1.get_text()

In [ ]:
import pytest
from solution import Replica

def test_get_causality_state():
    """Tests that a causality state can be retrieved."""
    r = Replica("A", "hello")
    state = r.get_causality_state()
    assert state is not None
    # The format is opaque, but it should be serializable for network transport
    import json
    try:
        json.dumps(state)
    except TypeError:
        pytest.fail("Causality state is not JSON serializable")

def test_compaction_does_not_alter_text():
    """Compacting should not change the user-visible text."""
    r1 = Replica("R1", "abc")
    r2 = Replica("R2", "abc")
    
    op1 = r1.local_delete(1, 1) # "ac"
    r2.apply_remote(op1)
    
    text_before = r1.get_text()
    
    # Compact r1 using the knowledge that r2 is fully up-to-date
    state1 = r1.get_causality_state()
    state2 = r2.get_causality_state()
    r1.compact([state1, state2])
    
    assert r1.get_text() == text_before

def test_compaction_prevents_applying_stale_ops():
    """
    After compaction, applying an old operation that refers to garbage-collected
    history should have no effect or be gracefully handled.
    """
    r1 = Replica("R1", "abc")
    r2 = Replica("R2", "abc")

    # op_del_b refers to 'b' which is about to be deleted and garbage collected
    op_del_b = r1.local_delete(1, 1)
    
    # r2 applies it, so it's fully caught up
    r2.apply_remote(op_del_b)
    
    # r1 compacts, knowing r2 is up to date. The tombstone for 'b' can be purged.
    r1.compact([r1.get_causality_state(), r2.get_causality_state()])

    # Concurrently, a third replica (which was out of sync) tries to edit 'b'.
    # This op is now causally stale.
    r3 = Replica("R3", "abc")
    op_ins_x = r3.local_insert(2, "x") # Tries to make "abxc"
    
    text_before_stale_op = r1.get_text()
    
    # r1 receives this very old operation.
    try:
        r1.apply_remote(op_ins_x)
    except Exception as e:
        pytest.fail(f"Applying a stale operation after compaction crashed: {e}")

    # The text should not have changed, as the anchor position for the op
    # should be long gone.
    assert r1.get_text() == text_before_stale_op


def test_no_compaction_if_replica_is_behind():
    """
    Tests that tombstones are preserved if one replica is not caught up.
    This is tested indirectly by checking if a dependent operation can still be applied.
    """
    r1 = Replica("R1", "abc")
    r2 = Replica("R2", "abc")
    r3 = Replica("R3", "abc") # This replica will stay offline

    # R1 deletes 'b', R2 sees it.
    op_del_b = r1.local_delete(1, 1)
    r2.apply_remote(op_del_b)
    
    # R1 tries to compact, but it only knows about R2's state. R3 is missing.
    # The implementation should be conservative and not purge 'b's tombstone.
    r1.compact([r1.get_causality_state(), r2.get_causality_state()])
    
    # R2 creates an op that depends on 'b's existence (conceptually).
    # It tries to insert 'y' after where 'b' was.
    op_ins_y = r2.local_insert(1, "y") # R2 text: "ayc"

    # R1 should be able to apply this op correctly, because it shouldn't have
    # fully purged the history related to the 'b' position yet.
    r1.apply_remote(op_ins_y)
    
    assert r1.get_text() == "ayc"

In [ ]:
import pytest
from solution import Replica, Operation

def test_operation_serialization_deserialization_insert():
    """
    Tests that an insert Operation can be converted to a dict and back,
    and the resulting object is equal to the original.
    """
    r = Replica("A", "hello")
    op_insert = r.local_insert(5, " world")
    
    # Serialize
    op_dict = op_insert.to_dict()
    
    # Check format
    assert isinstance(op_dict, dict)
    import json
    try:
        json.dumps(op_dict)
    except TypeError:
        pytest.fail("Serialized operation dict is not JSON serializable")

    # Deserialize
    op_reconstructed = Operation.from_dict(op_dict)
    
    assert isinstance(op_reconstructed, Operation)
    assert op_insert == op_reconstructed
    # A stronger check for implementations that might not define __eq__ properly
    assert op_reconstructed.to_dict() == op_dict

def test_operation_serialization_deserialization_delete():
    """
    Tests that a delete Operation can be converted to a dict and back,
    and the resulting object is equal to the original.
    """
    r = Replica("B", "remove text")
    op_delete = r.local_delete(0, 7)
    
    # Serialize
    op_dict = op_delete.to_dict()
    
    # Check format
    assert isinstance(op_dict, dict)
    
    # Deserialize
    op_reconstructed = Operation.from_dict(op_dict)
    
    assert isinstance(op_reconstructed, Operation)
    assert op_delete == op_reconstructed

def test_deserialized_op_can_be_applied():
    """
    Tests the full round trip: serialize, deserialize, and then apply
    the reconstructed operation to another replica.
    """
    r1 = Replica("R1", "abc")
    r2 = Replica("R2", "abc")
    
    # 1. R1 creates an op
    op1 = r1.local_insert(1, "x")
    
    # 2. Serialize and deserialize it (as if sent over a network)
    op1_dict = op1.to_dict()
    op1_reconstructed = Operation.from_dict(op1_dict)
    
    # 3. R2 applies the reconstructed op
    r2.apply_remote(op1_reconstructed)
    
    assert r2.get_text() == "axbc"

def test_serialization_is_lossless_for_convergence():
    """
    Ensures that serialization doesn't discard information needed for
    correct convergence in a concurrent editing scenario.
    """
    r1 = Replica("A", "ac")
    r2 = Replica("B", "ac")

    # Both insert at index 1 concurrently
    op1_raw = r1.local_insert(1, "b")
    op2_raw = r2.local_insert(1, "x")
    
    # Simulate network transit by serializing/deserializing
    op1 = Operation.from_dict(op1_raw.to_dict())
    op2 = Operation.from_dict(op2_raw.to_dict())
    
    r1.apply_remote(op2)
    r2.apply_remote(op1)
    
    # As in the original convergence test, the result must be identical
    # and deterministic. If serialization lost the tie-breaker info
    # (e.g. replica ID), this test might fail.
    assert r1.get_text() == r2.get_text()
    assert r1.get_text() in ("abxc", "axbc")